# 🔌 Model Integration: One Interface, Many Providers

## Learning Objectives
In this notebook, you will learn:
1. **The course's LLM factory** - what `get_openai_llm(model="gpt-4o-mini")` gives you, and why every notebook uses it
2. **`init_chat_model`** - one provider-agnostic constructor that swaps models with a string
3. **Provider classes** - when to reach for `ChatOpenAI` / `ChatGroq` / `ChatGoogleGenerativeAI` directly instead
4. **Streaming** - render tokens as they are generated with `.stream()` for a responsive UX
5. **Batching** - run independent prompts in parallel with `.batch()` and cap concurrency

## Where this fits
Module 03 (`03_react_with_create_agent.ipynb`) built agents with `create_agent`. This module opens up the parts an agent is made of: the chat model (this notebook), the messages it reads and writes (`02_messages.ipynb`), typed output (`03_structured_output.ipynb`), and the middleware that controls the loop (`04_middleware.ipynb`).

## Prerequisites
- `create_agent` basics — `03_react_with_create_agent.ipynb` (module 03)
- Dependencies installed with `uv sync` at the repository root (includes `langchain-groq` and `langchain-google-genai`)
- `OPENAI_API_KEY` in the project-root `.env` (required)
- Optional: `GROQ_API_KEY` for the Groq demo, `GOOGLE_API_KEY` for the Gemini demo — each demo is skipped cleanly when its key is missing

---
## 📦 Step 1: Environment Setup

Load the keys and report which optional providers are available (key names only, never values).

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY (and optionally GROQ_API_KEY / GOOGLE_API_KEY) from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

import os

for key in ["OPENAI_API_KEY", "GROQ_API_KEY", "GOOGLE_API_KEY"]:
    print(f"{'✅' if os.getenv(key) else '⚪'} {key} {'set' if os.getenv(key) else 'not set'}")

print("✅ Environment ready")

✅ OPENAI_API_KEY set
✅ GROQ_API_KEY set
✅ GOOGLE_API_KEY set
✅ Environment ready


In [2]:
# ============================================================================
# LLM INITIALIZATION: The course's default chat model
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

✅ LLM initialized: gpt-4o-mini


---
## 🏭 Part 1: The Course Default — `get_openai_llm`

Every notebook in this course creates its chat model with one line:
`get_openai_llm(model="gpt-4o-mini")`. It is a thin factory that returns a LangChain
`ChatOpenAI` with `temperature=0`, so the whole course switches model in one place. We always
name the model explicitly — the factory's own default is a different model.

What you get back is an ordinary LangChain chat model: `.invoke()` takes a string (or a list of
messages) and returns an `AIMessage`.

In [3]:
# ============================================================================
# FIRST INVOCATION: A single prompt in, one AIMessage out
# ============================================================================
response = llm.invoke("Hello! How are you?")

print(f"📋 type:    {type(response).__name__}")
print(f"🤖 content: {response.content}")

📋 type:    AIMessage
🤖 content: Hello! I'm just a program, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?


---
## 🌐 Part 2: `init_chat_model` — the Provider-Agnostic Way

`init_chat_model` builds a chat model from a **string**, inferring the provider from the model
name (or from an explicit `provider:model` prefix). This is the recommended default: switching
from OpenAI to Gemini becomes a one-string edit rather than an import change.

### Key Concepts:
- **Inferred provider**: `"gpt-4o-mini"` → OpenAI, because the name is unambiguous
- **Explicit provider**: `"openai:gpt-4o-mini"`, `"google_genai:gemini-2.5-flash"`,
  `"groq:openai/gpt-oss-20b"` — always safe, and required when a name is ambiguous
- **Uniform return type**: whatever you name, you get a `BaseChatModel` with the same
  `.invoke()` / `.stream()` / `.batch()` / `.bind_tools()` surface

In [4]:
# ============================================================================
# MODEL INIT: Provider-agnostic construction from a string
# ============================================================================
from langchain.chat_models import init_chat_model

model = init_chat_model("openai:gpt-4o-mini")  # explicit provider prefix

print(f"🤖 Loaded: {type(model).__name__} -> {model.model_name}")
print("🤖", model.invoke("In one sentence: what is a chat model?").content)

🤖 Loaded: ChatOpenAI -> gpt-4o-mini
🤖 A chat model is an artificial intelligence system designed to understand and generate human-like responses in natural language conversations.


### 2.1 🏭 Provider Classes Directly

`ChatOpenAI` and friends are still there and still supported. Use them when you need
constructor arguments that only that provider has (custom `base_url`, organisation IDs,
provider-specific tuning). Otherwise prefer `init_chat_model` — it keeps the notebook
portable.

> **Note**: all three routes — the factory, the string, the class — produce the same object type; `init_chat_model("openai:gpt-4o-mini")` simply
> constructs a `ChatOpenAI` for you under the hood.

In [5]:
# ============================================================================
# PROVIDER CLASS: ChatOpenAI constructed explicitly
# ============================================================================
from langchain_openai import ChatOpenAI

direct = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Three routes, one class:
for label, m in [("get_openai_llm", llm), ("init_chat_model", model), ("ChatOpenAI", direct)]:
    print(f"🏭 {label:<16} -> {type(m).__name__}({m.model_name})")

🏭 get_openai_llm   -> ChatOpenAI(gpt-4o-mini)
🏭 init_chat_model  -> ChatOpenAI(gpt-4o-mini)
🏭 ChatOpenAI       -> ChatOpenAI(gpt-4o-mini)


---
## ⚡ Part 3: Other Providers (Optional)

Swapping provider is a one-string edit: `"groq:..."` or `"google_genai:..."` instead of
`"openai:..."`. Everything downstream — `.invoke()`, `.stream()`, `.batch()`, `.bind_tools()`,
agents — stays the same. That symmetry is the point of the abstraction.

Both demos below are **optional**: each runs only when its key is set, and reports a failed call
instead of stopping the notebook.

### 3.1 ⚡ Groq

Groq serves open-weight models on custom inference hardware, so it is the usual choice when
latency matters more than frontier quality. Model ids are provider-specific
(`openai/gpt-oss-20b` here); list what your account can use with Groq's `models.list()`.

In [6]:
# ============================================================================
# GROQ (OPTIONAL): same init_chat_model call, different provider prefix
# ============================================================================
import os

if os.getenv("GROQ_API_KEY"):
    try:
        groq_model = init_chat_model("groq:openai/gpt-oss-20b")
        answer = groq_model.invoke("Why do parrots talk? Answer in two sentences.")
        print(f"⚡ {type(groq_model).__name__}: {answer.content}")
    except Exception as e:  # a demo: report, don't crash
        print(f"⚠️ Groq call failed: {type(e).__name__}: {str(e)[:200]}")
else:
    print("⚪ GROQ_API_KEY not set - skipping the Groq demo")

⚡ ChatGroq: Parrots talk by mimicking the sounds they hear, using their highly developed syrinx and vocal learning abilities to reproduce words and phrases. They do this to communicate with their social group, seek attention, and reinforce bonds, which is why they often repeat human speech.


### 3.2 ✨ Google Gemini

The same pattern with the `google_genai` prefix. The class route is
`from langchain_google_genai import ChatGoogleGenerativeAI` →
`ChatGoogleGenerativeAI(model="gemini-2.5-flash")`.

In [7]:
# ============================================================================
# GEMINI (OPTIONAL): same call again, google_genai prefix
# ============================================================================
if os.getenv("GOOGLE_API_KEY"):
    try:
        gemini_model = init_chat_model("google_genai:gemini-2.5-flash")
        answer = gemini_model.invoke("Why do parrots talk? Answer in two sentences.")
        print(f"✨ {type(gemini_model).__name__}: {answer.content}")
    except Exception as e:  # a demo: report, don't crash
        print(f"⚠️ Gemini call failed: {type(e).__name__}: {str(e)[:200]}")
else:
    print("⚪ GOOGLE_API_KEY not set - skipping the Gemini demo")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


✨ ChatGoogleGenerativeAI: Parrots talk due to their unique vocal anatomy, particularly a flexible tongue and a specialized vocal organ called a syrinx, which allows them to mimic a wide range of sounds. They primarily do this as a social behavior, using vocalizations to bond with their flock (including humans) and integrate into their environment by mirroring sounds they frequently hear.


---
## 🌊 Part 4: Streaming

Most models can emit output while it is still being generated. `.stream()` returns an iterator
of chunks instead of one finished message, which dramatically improves perceived latency on
long responses — the user sees the first words in a few hundred milliseconds rather than
staring at a spinner for ten seconds.

### Key Concepts:
- **`.invoke()`**: blocks, returns one complete `AIMessage`
- **`.stream()`**: yields `AIMessageChunk`s as they arrive; `chunk.text` is the new text
- **Same total cost**: streaming changes *when* you see tokens, not how many you pay for

In [8]:
# ============================================================================
# BASELINE: Blocking invoke — nothing renders until the whole answer is done
# ============================================================================
import time

PROMPT = "Write me a 150-word paragraph on Artificial Intelligence"

start = time.perf_counter()
full = llm.invoke(PROMPT)
print(f"⏱️ invoke: whole answer after {time.perf_counter() - start:.2f}s")
print(full.content[:200], "...")

⏱️ invoke: whole answer after 3.42s
Artificial Intelligence (AI) refers to the simulation of human intelligence processes by machines, particularly computer systems. These processes include learning, reasoning, problem-solving, percepti ...


In [9]:
# ============================================================================
# STREAMING: Same prompt, rendered chunk by chunk
# ============================================================================
# The "|" separator makes the chunk boundaries visible; drop it in real apps.
start, first_chunk_at = time.perf_counter(), None
for chunk in llm.stream(PROMPT):
    if first_chunk_at is None and chunk.text:
        first_chunk_at = time.perf_counter() - start
    print(chunk.text, end="|", flush=True)

print(f"\n\n⏱️ stream: first text after {first_chunk_at:.2f}s, done after {time.perf_counter() - start:.2f}s")

|Artificial| Intelligence| (|AI|)| refers| to| the| simulation| of| human| intelligence| processes| by| machines|,| particularly| computer| systems|.| These| processes| include| learning|,| reasoning|,| problem|-solving|,| perception|,| and| language| understanding|.| AI| can| be| categorized| into| two| main| types|:| narrow| AI|,| which| is| designed| for| specific| tasks|,| such| as| voice| recognition| or| recommendation| systems|,| and| general| AI|,| which| aims| to| perform| any| intellectual| task| that| a| human| can| do|.| The| rapid| advancements| in| AI| technologies|,| driven| by| increased| computational| power| and| vast| amounts| of| data|,| have| transformed| various| industries|,| from| healthcare| to| finance|.| AI| applications| enhance| efficiency|,| improve| decision|-making|,| and| enable| personalized| experiences|.| However|,| the| rise| of| AI| also| raises| ethical| concerns|,| including| job| displacement|,| privacy| issues|,| and| the| potential| for| biase

Compare the two timings: the total time is about the same (you pay for the same tokens), but
the streamed version shows its first words after a fraction of a second.

---
## 📦 Part 5: Batching

When you have several **independent** prompts, `.batch()` sends them in parallel instead of
looping. This cuts wall-clock time roughly by the degree of parallelism and lets the provider
schedule the work efficiently.

> **Note**: batching is for prompts that do not depend on each other. Anything conversational —
> where request *n+1* needs the answer to *n* — must stay sequential.

In [10]:
# ============================================================================
# BATCH: Three independent prompts, processed in parallel
# ============================================================================
responses = llm.batch([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?",
])

for i, response in enumerate(responses):
    print(f"\n📄 [{i}] {response.content[:200]}...")


📄 [0] Parrots have colorful feathers for several reasons, primarily related to their survival, communication, and mating. Here are some key factors:

1. **Camouflage**: In their natural habitats, which ofte...

📄 [1] Airplanes fly by generating lift, which counteracts the force of gravity, allowing them to ascend and remain airborne. The principles of flight can be explained through four fundamental forces: lift, ...

📄 [2] Quantum computing is a type of computation that takes advantage of the principles of quantum mechanics to process information in fundamentally different ways than classical computers. While classical ...


In [11]:
# ============================================================================
# BATCH: Capping parallelism to stay inside provider rate limits
# ============================================================================
responses = llm.batch(
    [
        "Why do parrots have colorful feathers?",
        "How do airplanes fly?",
        "What is quantum computing?",
    ],
    config={
        "max_concurrency": 2,  # at most 2 in-flight requests at a time
    },
)

print(f"✅ {len(responses)} answers, at most 2 requests in flight at once")

✅ 3 answers, at most 2 requests in flight at once


---
## 📝 Summary

In this notebook, we learned:

### 1. The Course Default
- **`get_openai_llm(model="gpt-4o-mini")`** returns a plain `ChatOpenAI` — one line to change the
  model for the whole course; always name the model explicitly

### 2. Two More Ways to Construct a Model
- **`init_chat_model("openai:gpt-4o-mini")`**: provider-agnostic, swap models by editing a string
- **`ChatOpenAI(...)` / `ChatGroq(...)`**: reach for these only when you need
  provider-specific constructor arguments
- **Explicit prefixes** (`"openai:"`, `"google_genai:"`, `"groq:"`) remove all ambiguity

### 3. One Interface, Many Providers
- **Uniform surface**: every provider returns a `BaseChatModel` with the same
  `.invoke()` / `.stream()` / `.batch()` / `.bind_tools()` methods
- **Model ids are provider-specific** and change over time — check the provider's model list

### 4. Streaming
- **`.stream()`** yields chunks; `chunk.text` holds the incremental text
- **UX, not cost**: same token count, far better time to first word

### 5. Batching
- **`.batch()`** runs independent prompts in parallel
- **`config={"max_concurrency": n}`** keeps you inside provider rate limits

### Next Steps
- **`02_messages.ipynb`** — the message types (`System`, `Human`, `AI`, `Tool`), metadata, token
  usage, and the message trail an agent run returns